# PyCram KnowRob Demo
This demo shows how to use PyCram with the KnowRob reasoning framework to infer knowledge about the world on demand. The scenario is a PR2 robot in the IAI apartment lab which has the task of setting the table for a breakfast. To achieve this the robot has to find a milk bottle in the apartment and transport it to the table as well as a bowl and a spoon.

The robot is only given the types of the objects and the final pose on the table and has to figure out the exact objects and their poses by itself.

We will start the demo by setting up the scene with the PR2 robot, the apartment and the milk bottle. Afterwards, we will bring the robot into an initial state by parking its arms and moving the torso to a high position.

In [1]:
%%bash --bg
export DISABLE_ROS1_EOL_WARNINGS=1
rviz --fullscreen -d /home/jovyan/workspace/ros/src/knowrob_cram_demo/binder/rviz_configs/knowrob_cram_setup.rviz

In [2]:
%%capture out
from pycram.datastructures.pose import PoseStamped
from pycram.language import SequentialPlan
from pycram.process_module import simulated_robot
from pycram.ros_utils.viz_marker_publisher import VizMarkerPublisher
from pycrap.ontologies import Robot, Apartment, Milk, Bowl, Spoon, Cereal
from pycram.world_concepts.world_object import Object

from pycram.worlds.bullet_world import BulletWorld

world = BulletWorld()

pr2 = Object("pr2", Robot, "pr2.urdf", pose=PoseStamped.from_list([1, 1, 0]))
apartment = Object("apartment", Apartment, "apartment.urdf")
milk = Object("milk", Milk, "milk.stl", pose=PoseStamped.from_list([0.5, 2.5, 1.2]))
bowl = Object("bowl", Bowl, "bowl.stl", pose=PoseStamped.from_list([3, 2.1, 0.98]))
spoon = Object("spoon", Spoon, "spoon.stl", pose=PoseStamped.from_list([3, 2.2, 0.955], [0, 0, 1, 0]))
cereal = Object("cereal", Cereal, "breakfast_cereal.stl", pose=PoseStamped.from_list([3, 2.3, 1.04]))

v = VizMarkerPublisher()

[WARN] [1750761321.502988147] [pycram]: Import for control_msgs for gripper in Multiverse failed
pycram_bullet build time: Sep  5 2024 09:04:59
[INFO] [1750761321.963372550] [pycram]: Could not initialize ur5e description as Multiverse resources path not found.
[INFO] [1750761322.417614732] [pycram]: Found file plane.urdf in /home/jdech/workspace/ros/src/pycram-1/resources/objects/plane.urdf
[INFO] [1750761322.430522851] [pycram]: Found file pr2.urdf in /home/jdech/workspace/ros/src/pycram-1/resources/robots/pr2.urdf
[INFO] [1750761322.640752486] [pycram]: Found file apartment.urdf in /home/jdech/workspace/ros/src/pycram-1/resources/objects/apartment.urdf
[INFO] [1750761324.167767888] [pycram]: Found file milk.stl in /home/jdech/workspace/ros/src/pycram-1/resources/objects/milk.stl
[INFO] [1750761324.174745068] [pycram]: Found file bowl.stl in /home/jdech/workspace/ros/src/pycram-1/resources/objects/bowl.stl
[INFO] [1750761324.187065040] [pycram]: Found file spoon.stl in /home/jdech/wo

In [3]:
from pycram.datastructures.enums import Arms, TorsoState
from pycram.designators.action_designator import ParkArmsActionDescription, MoveTorsoActionDescription

with simulated_robot:
    SequentialPlan(ParkArmsActionDescription(Arms.BOTH),
    MoveTorsoActionDescription(TorsoState.HIGH)).perform()

Now it is time for the actual task. The robot has to find the milk bottle and transport it to the table, the only inputs are the type of the object and the final pose on the table. The robot will use the KnowRob reasoning framework to infer the exact object and its pose in the apartment.

In [4]:
%%capture out
from pycram.designators.location_designator import KnowledgeLocation
from pycram.designators.object_designator import ResolutionStrategyObject
from pycram.designators.action_designator import TransportActionDescription, SearchActionDescription

with simulated_robot:
    TransportActionDescription(
        ResolutionStrategyObject(strategy=SearchActionDescription(KnowledgeLocation(Milk), Milk)),
        PoseStamped.from_list([3.1, 1.95, 1.05], [0, 0, 1, 1])).perform()
